In [1]:
from pyspark.sql import SparkSession

In [2]:
spark =    SparkSession.builder \
    .appName("day10") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access",  "true") \
    .config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.catalogImplementation",        "hive") \
    .config("hive.metastore.uris",                    "thrift://metastore:9083") \
    .config("spark.eventLog.enabled",                 "true") \
    .config("spark.eventLog.dir",                     "s3a://spark-event/") \
    .config("spark.sql.shuffle.partitions",           str(4)) \
    .config("spark.sql.adaptive.enabled",             "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
spark.sparkContext.setLogLevel("WARN")

transformation

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
# ── HIVE CHECK ────────────────────────────────────────────────────────────────
# local_db is auto-created by init-hive-db container on first startup
print("\n── Hive databases (local_db should appear) ────────────────────────────")
spark.sql("SHOW DATABASES").show()


── Hive databases (local_db should appear) ────────────────────────────
+---------+
|namespace|
+---------+
|  default|
| local_db|
+---------+



In [6]:
spark.sql("SHOW TABLES IN local_db").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
| local_db|sample_hive_table|      false|
+---------+-----------------+-----------+



── LOAD DATA ─────────────────────────────────────────────────────────────────
- Prefer Hive table — no file path needed, schema already known

In [7]:

try:
    df = spark.sql("SELECT * FROM local_db.sample_hive_table")
    print("[INFO] Loaded from Hive: local_db.sample_hive_table")
except Exception:
    df = spark.read.parquet("s3a://spark-warehouse/raw_data/save_from_notebook")
    print("[WARN] Hive table not found — loaded from Parquet. Run Day 9 first.")
 
taxi_zone = spark.read \
    .option("header", "true").option("inferSchema", "true") \
    .csv("s3a://metadata-rawdata/taxi_zone_lookup.csv")
 
 
print(f"taxi : {df.count()} rows")
print(f"taxi_zone : {taxi_zone.count()} rows")
 

[INFO] Loaded from Hive: local_db.sample_hive_table
taxi : 300000 rows
taxi_zone : 265 rows


In [8]:
# ── SECTION 1: GROUPBY + AGGREGATION ─────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 1 — GroupBy Aggregation")
print("=" * 60)


SECTION 1 — GroupBy Aggregation


In [9]:
date_stats = df.groupBy("tpep_pickup_date").agg(
    F.count("VendorID").alias("Vendor count"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.min("fare_amount").alias("min_fare"),
    F.round(F.sum("fare_amount"), 2).alias("total_fare"),
)
date_stats.orderBy(F.col("avg_fare").desc()).show()
 
# Multi-column groupBy — city breakdown per department
print("── Zone × Date headcount ──────────────────────────────")
df.groupBy("tpep_pickup_date", "PULocationID").agg(
    F.count("PULocationID").alias("zone_count")
).orderBy("tpep_pickup_date", F.col("zone_count").desc()).show(20)

+----------------+------------+--------+--------+--------+----------+
|tpep_pickup_date|Vendor count|avg_fare|max_fare|min_fare|total_fare|
+----------------+------------+--------+--------+--------+----------+
|      2025-01-02|       84832|    19.1|   423.7|  -336.2|1620005.97|
|      2025-01-03|       91250|   18.17|   670.1|  -175.2| 1657619.5|
|      2025-01-04|       33730|   17.67|   325.0|  -136.0| 596168.73|
|      2025-01-01|       90188|   17.66|   826.2|  -826.2|1593061.51|
+----------------+------------+--------+--------+--------+----------+

── Zone × Date headcount ──────────────────────────────
+----------------+------------+----------+
|tpep_pickup_date|PULocationID|zone_count|
+----------------+------------+----------+
|      2025-01-01|         132|      5416|
|      2025-01-01|         161|      3398|
|      2025-01-01|         186|      3138|
|      2025-01-01|          79|      3125|
|      2025-01-01|         230|      2947|
|      2025-01-01|          48|      29

In [10]:
# ── SECTION 2: JOINS ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 2 — Joins")
print("=" * 60)


SECTION 2 — Joins


In [11]:
# zone has 265 rows — always broadcast small dimension tables.
# Broadcast sends the table to every executor so they never need to shuffle.
# Without broadcast: both tables shuffle across the network (SortMergeJoin).
# With broadcast: only the large table moves, small table is local (BroadcastHashJoin).

taxi_zone_PU = taxi_zone.alias("PU")
taxi_zone_DO = taxi_zone.alias("DO")


enriched = df.join(
    F.broadcast(taxi_zone_PU),
    df["PULocationID"] == taxi_zone_PU["PU.LocationID"],
    how="left"
)

enriched = enriched.join(
    F.broadcast(taxi_zone_DO),
    df["DOLocationID"] == taxi_zone_DO["DO.LocationID"],
    how="left"
)

 


In [12]:
print("── Confirm BroadcastHashJoin in physical plan ─────────")
# Look for 'BroadcastHashJoin' not 'SortMergeJoin'
enriched.explain()
 
print("── Enriched sample ────────────────────────────────────")
enriched.select("tpep_pickup_date", "PU.Zone", "DO.Zone", "PU.Borough", "DO.Borough") \
        .orderBy("tpep_pickup_date").show(10)
 
# Join employees to sales (inner — only employees who have sales records)
#emp_sales = employees.join(sales, on="emp_id", how="inner")
#print(f"\nEmployees with at least one sales record: {emp_sales.count()} rows")

── Confirm BroadcastHashJoin in physical plan ─────────
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [DOLocationID#48], [LocationID#330], LeftOuter, BuildRight, false
   :- BroadcastHashJoin [PULocationID#47], [LocationID#102], LeftOuter, BuildRight, false
   :  :- FileScan parquet spark_catalog.local_db.sample_hive_table[VendorID#40,tpep_pickup_datetime#41,tpep_dropoff_datetime#42,passenger_count#43L,trip_distance#44,RatecodeID#45L,store_and_fwd_flag#46,PULocationID#47,DOLocationID#48,payment_type#49L,fare_amount#50,extra#51,mta_tax#52,tip_amount#53,improvement_surcharge#54,total_amount#55,congestion_surcharge#56,Airport_fee#57,cbd_congestion_fee#58,total_amount_thb#59,is_valid_passenger_count#60,tpep_pickup_date#61] Batched: true, DataFilters: [], Format: Parquet, Location: CatalogFileIndex(1 paths)[s3a://spark-warehouse/hive/local_db/sample_hive_table], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:times

In [13]:
# ── SECTION 3: WINDOW FUNCTIONS ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 3 — Window Functions")
print("=" * 60)


SECTION 3 — Window Functions


In [15]:
# 3a.
# rank() gives gaps (1,2,2,4), dense_rank() doesn't (1,2,2,3), row_number() is unique (1,2,3,4)
date_window = Window.partitionBy("tpep_pickup_date").orderBy(F.col("total_amount").desc())
 
ranked = enriched \
    .withColumn("total_amount_rank",  F.rank().over(date_window)) \
    .withColumn("dense_rank",   F.dense_rank().over(date_window)) \
    .withColumn("row_number",   F.row_number().over(date_window))
 
print("── Top 2 earners per date ───────────────────────")
ranked.filter(F.col("total_amount_rank") <= 2) \
      .select("tpep_pickup_date", "PU.Zone", "PU.Borough", "total_amount", "total_amount_rank", "dense_rank") \
      .orderBy("tpep_pickup_date", "total_amount_rank") \
      .show()
 


── Top 2 earners per date ───────────────────────
+----------------+--------------------+---------+------------+-----------------+----------+
|tpep_pickup_date|                Zone|  Borough|total_amount|total_amount_rank|dense_rank|
+----------------+--------------------+---------+------------+-----------------+----------+
|      2025-01-01|         JFK Airport|   Queens|      865.39|                1|         1|
|      2025-01-01|           Glen Oaks|   Queens|      794.82|                2|         2|
|      2025-01-02|         JFK Airport|   Queens|      480.26|                1|         1|
|      2025-01-02|         JFK Airport|   Queens|      477.21|                2|         2|
|      2025-01-03|         JFK Airport|   Queens|      702.11|                1|         1|
|      2025-01-03|         JFK Airport|   Queens|      607.01|                2|         2|
|      2025-01-04|Greenwich Village...|Manhattan|      427.05|                1|         1|
|      2025-01-04|    South Oz

In [18]:
# 3b.
# This is impossible with a single groupBy — you'd need a self-join.
date_avg_window = Window.partitionBy("tpep_pickup_date")
 
print("── Salary vs dept average (all rows preserved) ────────")
enriched \
    .withColumn("total_amount_avg",    F.round(F.avg("total_amount").over(date_avg_window), 2)) \
    .withColumn("pct_of_total_amount", F.round(
        F.col("total_amount") / F.sum("total_amount").over(date_avg_window) * 100, 1)
    ) \
    .select("tpep_pickup_date", "PU.Zone", "PU.Borough", "total_amount", "total_amount_avg", "pct_of_total_amount") \
    .orderBy("tpep_pickup_date", F.col("total_amount").desc()) \
    .show(15)
 
# orderBy("tpep_pickup_date", F.col("total_amount").desc()) -> orderBy(string, columnObject) : columnObject can be customeized with other function like desc()

── Salary vs dept average (all rows preserved) ────────
+----------------+--------------------+---------+------------+----------------+-------------------+
|tpep_pickup_date|                Zone|  Borough|total_amount|total_amount_avg|pct_of_total_amount|
+----------------+--------------------+---------+------------+----------------+-------------------+
|      2025-01-01|         JFK Airport|   Queens|      865.39|           25.41|                0.0|
|      2025-01-01|           Glen Oaks|   Queens|      794.82|           25.41|                0.0|
|      2025-01-01|      Outside of NYC|      N/A|       701.0|           25.41|                0.0|
|      2025-01-01|      Outside of NYC|      N/A|       701.0|           25.41|                0.0|
|      2025-01-01|   East Harlem North|Manhattan|      525.36|           25.41|                0.0|
|      2025-01-01|         JFK Airport|   Queens|      516.13|           25.41|                0.0|
|      2025-01-01|        Clinton East|Manha

In [21]:
# 3c. Time-series: lag, month-over-month change (date-over-date in this case), running total
print("── Month-over-month sales change (lag) ────────────────")
time_window = Window.partitionBy("PU.Zone", "PU.Borough" ).orderBy("tpep_pickup_datetime")
 
zone_amount = enriched \
    .withColumn("prev_amount",    F.lag("total_amount", 1).over(time_window)) \
    .withColumn("dod_change",    F.round( F.col("total_amount") - F.lag("total_amount", 1).over(time_window), 2) ) \
    .withColumn("accumulating_total", F.round( F.sum("total_amount").over( time_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)), 2))

zone_amount.select("PU.Zone", "PU.Borough", "tpep_pickup_datetime", "total_amount", "prev_amount","dod_change", "accumulating_total") \
    .orderBy("PU.Zone", "PU.Borough", "tpep_pickup_date").show(20)
 


── Month-over-month sales change (lag) ────────────────
+--------------------+---------+--------------------+------------+-----------+----------+------------------+
|                Zone|  Borough|tpep_pickup_datetime|total_amount|prev_amount|dod_change|accumulating_total|
+--------------------+---------+--------------------+------------+-----------+----------+------------------+
|Allerton/Pelham G...|    Bronx| 2025-01-01 01:49:10|        22.7|       NULL|      NULL|              22.7|
|Allerton/Pelham G...|    Bronx| 2025-01-01 03:23:20|        2.92|       22.7|    -19.78|             25.62|
|Allerton/Pelham G...|    Bronx| 2025-01-01 04:50:00|      -11.07|       2.92|    -13.99|             14.55|
|Allerton/Pelham G...|    Bronx| 2025-01-01 06:55:21|       -2.61|     -11.07|      8.46|             11.94|
|Allerton/Pelham G...|    Bronx| 2025-01-01 08:29:15|        18.0|      -2.61|     20.61|             29.94|
|Allerton/Pelham G...|    Bronx| 2025-01-01 08:54:24|        33.0|      

In [24]:
# 3d. Rolling 3-month average
print("── 3-month rolling average ─────────────────────────────")
rolling_window = Window.partitionBy("PU.Zone", "PU.Borough") \
                       .orderBy("tpep_pickup_datetime") \
                       .rowsBetween(-2, Window.currentRow)
 
enriched.select("PU.Zone", "PU.Borough", "tpep_pickup_datetime", "total_amount") \
    .withColumn("rolling_3m_avg", F.round(F.avg("total_amount").over(rolling_window), 2)) \
     .orderBy("PU.Zone", "PU.Borough", "tpep_pickup_datetime").show(20)

── 3-month rolling average ─────────────────────────────
+--------------------+---------+--------------------+------------+--------------+
|                Zone|  Borough|tpep_pickup_datetime|total_amount|rolling_3m_avg|
+--------------------+---------+--------------------+------------+--------------+
|Allerton/Pelham G...|    Bronx| 2025-01-01 01:49:10|        22.7|          22.7|
|Allerton/Pelham G...|    Bronx| 2025-01-01 03:23:20|        2.92|         12.81|
|Allerton/Pelham G...|    Bronx| 2025-01-01 04:50:00|      -11.07|          4.85|
|Allerton/Pelham G...|    Bronx| 2025-01-01 06:55:21|       -2.61|         -3.59|
|Allerton/Pelham G...|    Bronx| 2025-01-01 08:29:15|        18.0|          1.44|
|Allerton/Pelham G...|    Bronx| 2025-01-01 08:54:24|        33.0|         16.13|
|Allerton/Pelham G...|    Bronx| 2025-01-01 09:40:38|        17.0|         22.67|
|Allerton/Pelham G...|    Bronx| 2025-01-01 15:31:24|       -1.61|         16.13|
|Allerton/Pelham G...|    Bronx| 2025-01-

── WRITE ANALYTICS OUTPUT ────────────────────────────────────────────────────

In [25]:
date_stats.write.mode("overwrite").parquet("s3a://raw-data/output/date_stats/")
print("\n✓ date_stats written to minio: s3a://raw-data/output/date_stats/")


✓ date_stats written to minio: s3a://raw-data/output/date_stats/


In [16]:
spark.stop()
print("Done — check :18080 for job history")

Done — check :18080 for job history


**exercise1** sortMergeJoin -not using broadcastjoin

In [26]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)


In [27]:
no_broadcast = df.join(taxi_zone_PU, df["PULocationID"] == taxi_zone_PU["PU.LocationID"], how="left")
no_broadcast = no_broadcast.join(taxi_zone_DO, df["DOLocationID"] == taxi_zone_DO["DO.LocationID"], how="left")


no_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [DOLocationID#48], [LocationID#1532], LeftOuter
   :- Sort [DOLocationID#48 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(DOLocationID#48, 4), ENSURE_REQUIREMENTS, [plan_id=2326]
   :     +- SortMergeJoin [PULocationID#47], [LocationID#102], LeftOuter
   :        :- Sort [PULocationID#47 ASC NULLS FIRST], false, 0
   :        :  +- Exchange hashpartitioning(PULocationID#47, 4), ENSURE_REQUIREMENTS, [plan_id=2319]
   :        :     +- FileScan parquet spark_catalog.local_db.sample_hive_table[VendorID#40,tpep_pickup_datetime#41,tpep_dropoff_datetime#42,passenger_count#43L,trip_distance#44,RatecodeID#45L,store_and_fwd_flag#46,PULocationID#47,DOLocationID#48,payment_type#49L,fare_amount#50,extra#51,mta_tax#52,tip_amount#53,improvement_surcharge#54,total_amount#55,congestion_surcharge#56,Airport_fee#57,cbd_congestion_fee#58,total_amount_thb#59,is_valid_passenger_count#60,tpep_pickup_date#61] Batched: tr

In [28]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760) # file size

**exercise2**

group by vs window function like in sql
- group by reduce a number of rows
- window functions still keeps a number of rows

**exercise3**    

ROW_NUMBER deduplication (CDC pattern — appears in real interviews)    
In Change Data Capture pipelines, the same record arrives multiple times with different updated_at timestamps. Strategy: keep the latest version.

In [29]:
from pyspark.sql import Row

In [30]:
# Simulate duplicate records with different hire_dates
dupes = taxi_zone.limit(10).withColumn("service_zone", F.concat(F.col("service_zone"), F.lit(" 12")))
combined = taxi_zone.union(dupes)
print(f"With dupes: {combined.count()} rows")


With dupes: 275 rows


In [33]:
# Deduplicate using ROW_NUMBER — keep the LATEST version per emp_id
dedup_window = Window.partitionBy("LocationID").orderBy(F.col("service_zone").asc())
combined.withColumn("rn", F.row_number().over(dedup_window)).show()


+----------+-------------+--------------------+--------------+---+
|LocationID|      Borough|                Zone|  service_zone| rn|
+----------+-------------+--------------------+--------------+---+
|         1|          EWR|      Newark Airport|           EWR|  1|
|         1|          EWR|      Newark Airport|        EWR 12|  2|
|         2|       Queens|         Jamaica Bay|     Boro Zone|  1|
|         2|       Queens|         Jamaica Bay|  Boro Zone 12|  2|
|         3|        Bronx|Allerton/Pelham G...|     Boro Zone|  1|
|         3|        Bronx|Allerton/Pelham G...|  Boro Zone 12|  2|
|         4|    Manhattan|       Alphabet City|   Yellow Zone|  1|
|         4|    Manhattan|       Alphabet City|Yellow Zone 12|  2|
|         5|Staten Island|       Arden Heights|     Boro Zone|  1|
|         5|Staten Island|       Arden Heights|  Boro Zone 12|  2|
|         6|Staten Island|Arrochar/Fort Wad...|     Boro Zone|  1|
|         6|Staten Island|Arrochar/Fort Wad...|  Boro Zone 12|

In [34]:
deduped = combined \
  .withColumn("rn", F.row_number().over(dedup_window)) \
  .filter(F.col("rn") == 1) \
  .drop("rn")
print(f"After dedup: {deduped.count()} rows")

After dedup: 265 rows


Why is this better than dropDuplicates(["**"])?     
dropDuplicates keeps an arbitrary row — you can't control which one.     
ROW_NUMBER lets you define "latest" explicitly. Critical for CDC.

**exercise4**    
Save window results as a Hive table

In [37]:
ranked.show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+----------------+------------------------+----------------+----------+-------+-----------+------------+----------+-------+--------------+------------+-----------------+----------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_amount_thb|is_valid_passenger_count|tpep_pickup_date|LocationID|Borough|       Zone|service_zone|LocationID|Borough|          Zone|service_zone|total_amount_rank|dense_rank|row_number|
+--------+--------------------+---------------------+-------------

In [40]:
new_columns = [
    "PU_" + c if i > 21 and i < 26  else ("DO_" + c if i >= 26 and i <30 else c)
    for i, c in enumerate(ranked.columns)
]

In [41]:
new_columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'total_amount_thb',
 'is_valid_passenger_count',
 'tpep_pickup_date',
 'PU_LocationID',
 'PU_Borough',
 'PU_Zone',
 'PU_service_zone',
 'DO_LocationID',
 'DO_Borough',
 'DO_Zone',
 'DO_service_zone',
 'total_amount_rank',
 'dense_rank',
 'row_number']

In [42]:
ranked = ranked.toDF(*new_columns)
ranked.show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+----------------+------------------------+----------------+-------------+----------+-----------+---------------+-------------+----------+--------------+---------------+-----------------+----------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_amount_thb|is_valid_passenger_count|tpep_pickup_date|PU_LocationID|PU_Borough|    PU_Zone|PU_service_zone|DO_LocationID|DO_Borough|       DO_Zone|DO_service_zone|total_amount_rank|dense_rank|row_number|
+--------+--------------------

In [43]:
ranked.write.mode("overwrite") \
      .format("parquet").saveAsTable("local_db.total_amount_ranked")

In [44]:
spark.sql("SHOW TABLES IN local_db").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
| local_db|  sample_hive_table|      false|
| local_db|total_amount_ranked|      false|
+---------+-------------------+-----------+

